In [1]:
%load_ext watermark
%watermark -a 'Sebastian Raschka' -v -p torch

Author: Sebastian Raschka

Python implementation: CPython
Python version       : 3.12.2
IPython version      : 9.11.0

torch: 2.4.0



# Understanding Onehot Encoding and Cross Entropy in PyTorch

In [2]:
import torch

## Onehot Encoding

In [3]:
def to_onehot(y, num_classes):
    y_onehot = torch.zeros(y.size(0), num_classes)
    y_onehot.scatter_(1, y.view(-1, 1).long(), 1).float()
    return y_onehot

y = torch.tensor([0, 1, 2, 2])

y_enc = to_onehot(y, 3)

print('one-hot encoding:\n', y_enc)

one-hot encoding:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 0., 1.]])


## Softmax

Suppose we have some net inputs Z, where each row is one training example:

In [4]:
Z = torch.tensor( [[-0.3,  -0.5, -0.5],
                   [-0.4,  -0.1, -0.5],
                   [-0.3,  -0.94, -0.5],
                   [-0.99, -0.88, -0.5]])

Z

tensor([[-0.3000, -0.5000, -0.5000],
        [-0.4000, -0.1000, -0.5000],
        [-0.3000, -0.9400, -0.5000],
        [-0.9900, -0.8800, -0.5000]])

Next, we convert them to "probabilities" via softmax:

$$P(y=j \mid z^{(i)}) = \sigma_{\text{softmax}}(z^{(i)}) = \frac{e^{z^{(i)}}}{\sum_{j=0}^{k} e^{z_{k}^{(i)}}}.$$

In [5]:
def softmax(z):
    return (torch.exp(z.t()) / torch.sum(torch.exp(z), dim=1)).t()

smax = softmax(Z)
print('softmax:\n', smax)

softmax:
 tensor([[0.3792, 0.3104, 0.3104],
        [0.3072, 0.4147, 0.2780],
        [0.4263, 0.2248, 0.3490],
        [0.2668, 0.2978, 0.4354]])


The probabilties can then be converted back to class labels based on the largest probability in each row:

In [6]:
def to_classlabel(z):
    return torch.argmax(z, dim=1)

print('predicted class labels: ', to_classlabel(smax))
print('true class labels: ', to_classlabel(y_enc))

predicted class labels:  tensor([0, 1, 0, 2])
true class labels:  tensor([0, 1, 2, 2])


## Cross Entropy

Next, we compute the cross entropy for each training example:

$$\mathcal{L}(\mathbf{W}; \mathbf{b}) = \frac{1}{n} \sum_{i=1}^{n} H(T_i, O_i),$$

$$H(T_i, O_i) = -\sum_m T_i \cdot log(O_i).$$

In [7]:
def cross_entropy(softmax, y_target):
    return - torch.sum(torch.log(softmax) * (y_target), dim=1)

xent = cross_entropy(smax, y_enc)
print('Cross Entropy:', xent)

Cross Entropy: tensor([0.9698, 0.8801, 1.0527, 0.8314])


## In PyTorch

In [8]:
import torch.nn.functional as F

Note that `nll_loss` takes log(softmax) as input:

In [9]:
F.nll_loss(torch.log(smax), y, reduction='none')

tensor([0.9698, 0.8801, 1.0527, 0.8314])

Note that `cross_entropy` takes logits as input:

In [10]:
F.cross_entropy(Z, y, reduction='none')

tensor([0.9698, 0.8801, 1.0527, 0.8314])

### Defaults

By default, nll_loss & cross_entropy are already returning the average over training examples, which is useful for stability during optimization.

In [11]:
F.cross_entropy(Z, y)

tensor(0.9335)

In [13]:
torch.mean(cross_entropy(smax, y_enc))

tensor(0.9335)

In [14]:
import torch

In [17]:
tensor0D = torch.tensor(1)
tensor0D

tensor(1)

In [18]:
tensor1D = torch.tensor([1,2,3])
tensor1D

tensor([1, 2, 3])

In [42]:
tensor2D = torch.tensor([[1,2,3],[4,5,6]])
tensor2D

tensor([[1, 2, 3],
        [4, 5, 6]])

In [41]:
tensor3D = torch.tensor([[[1,2,3],[4,5,6]],[[7,8,9],[0,0,0]]])
tensor3D

tensor([[[1, 2, 3],
         [4, 5, 6]],

        [[7, 8, 9],
         [0, 0, 0]]])

In [26]:
tensor1D.dtype

torch.int64

In [27]:
floatvec = torch.tensor([1.0,2.0,3.0])

In [28]:
floatvec.dtype

torch.float32

In [33]:
floatvec64 = floatvec.to(torch.float64)

In [34]:
floatvec64.dtype

torch.float64

In [35]:
floatvec32 = floatvec64.to(torch.float32)

In [37]:
floatvec32.dtype

torch.float32

In [43]:
print(tensor2D.shape)

torch.Size([2, 3])


In [45]:
print(tensor2D.reshape(3,2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [46]:
tensor2D.matmul(tensor2D.T)

tensor([[14, 32],
        [32, 77]])

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [48]:
#torch的nn是神经网络,functional存放了很多的神经网络的函数
import torch.nn.functional as F
from torch.autograd import grad

In [49]:
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
#requires_grad告诉其需要求导,pytorch默默记录其参与了哪些运算方便求导
#[]是1D张量
w1 = torch.tensor([2.2],requires_grad = True)
b = torch.tensor([0.0],requires_grad = True)

In [50]:
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a,y)

In [51]:
#retain_graph如果没有这个参数,计算完梯度后计算图会丢弃,但是b还需要
#所以这里返回计算图
#w和b参与了一个计算所以用一个计算图
    #b之前配置了需要计算梯度只是代表计算图记录需要b的grad的必要信息
    #但是这里的grad()如果不配置retain_graph那么计算图将被删除,所以需要保留
grad_L_w1 = grad(loss,w1,retain_graph = True)
grad_L_b = grad(loss,b,retain_graph = True)

In [52]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)
